In [0]:
%sql

INSERT OVERWRITE proyecto_final.gold.fact_entidades (
    entidad_id,
    categoria_id,
    barrio_id,
    comuna_id,
    latitud,
    longitud,
    es_generador_trafico,
    es_competencia
)
WITH entidades AS (

    SELECT tipo_entidad, tipo_escuela AS subcategoria, nombre_establecimiento AS nombre_entidad, direccion, longitud,  latitud, coordenadas, barrio, comuna FROM proyecto_final.silver.escuelas
    UNION ALL
    SELECT tipo_entidad, especialidad AS subcategoria, nombre_hospital, direccion, longitud, latitud, coordenadas, barrio, comuna FROM proyecto_final.silver.hospitales
    UNION ALL
    SELECT tipo_entidad, categoria AS subcategoria, nombre, direccion, longitud, latitud, coordenadas, barrio, comuna FROM proyecto_final.silver.kioscos
    UNION ALL
    SELECT tipo_entidad, unidad_academica AS subcategoria, nombre_universidad, direccion, longitud, latitud, coordenadas, barrio, comuna FROM proyecto_final.silver.universidades
)
SELECT 
    dim_e.id AS entidad_id,
    dim_c.id AS categoria_id,
    dim_b.id AS barrio_id,
    dim_co.id AS comuna_id,
    
    e.latitud,
    e.longitud,
    
    CASE WHEN e.tipo_entidad IN ('escuela', 'hospital', 'universidad') THEN 1 ELSE 0 END AS es_generador_trafico,
    CASE WHEN e.tipo_entidad = 'kiosco' THEN 1 ELSE 0 END AS es_competencia

FROM entidades e
LEFT JOIN proyecto_final.gold.dim_entidad dim_e 
    ON e.nombre_entidad = dim_e.nombre_entidad AND e.coordenadas = dim_e.coordenadas
LEFT JOIN proyecto_final.gold.dim_categoria dim_c 
    ON e.tipo_entidad = dim_c.tipo_entidad AND COALESCE(e.subcategoria, '') = COALESCE(dim_c.subcategoria, '')
LEFT JOIN proyecto_final.gold.dim_barrios dim_b 
    ON e.barrio = dim_b.barrio_nombre
LEFT JOIN proyecto_final.gold.dim_comunas dim_co 
    ON e.comuna = dim_co.nro_comuna;

In [0]:
select * from proyecto_final.gold.fact_entidades